In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

from pyspark_data_provenance import (
    data_provenance_enabled, 
    build_data_provenance_session
)

In [3]:
from pyspark.sql import SparkSession


spark = (
    # SparkSession
    # .builder
    build_data_provenance_session()
    .appName("data-provenance-notebook")
    .getOrCreate()
)


26/04/27 10:13:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
print(spark.conf.get("spark.provenance.enabled", "false"))

with data_provenance_enabled(spark):
    print(spark.conf.get("spark.provenance.enabled", "false"))

print(spark.conf.get("spark.provenance.enabled", "false"))

false
true
false


## Create toy dataframes

In [5]:
from datetime import date

df = spark.createDataFrame([
    ("A", date(2026, 1, 15), 10.0, 90),
    ("A", date(2026, 1, 16), 10.0, 120),
    ("A", date(2026, 1, 17), 5.0, 300),
    ("B", date(2026, 1, 15), 100.0, 20),
    ("B", date(2026, 1, 16), 100.0, 30),
    ("C", date(2026, 1, 17), 80.0, 60),
    ("F", date(2026, 1, 16), 50.0, 70)
], ["product", "date", "price", "quantity"]
)
df.printSchema()

df.toPandas()

root
 |-- product: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: long (nullable = true)



,product,date,price,quantity
0,A,2026-01-15,10.0,90
1,A,2026-01-16,10.0,120
2,A,2026-01-17,5.0,300
3,B,2026-01-15,100.0,20
4,B,2026-01-16,100.0,30
5,C,2026-01-17,80.0,60
6,F,2026-01-16,50.0,70


In [6]:
df2 = spark.createDataFrame([
    ("A", "Bike"),
    ("B", "Handball"),
    ("C", "Bike"),
    ("D", "Handball"),
    ("E", "Running")
],["letter","labell"]
)

df2.printSchema()
df2.toPandas()



root
 |-- letter: string (nullable = true)
 |-- labell: string (nullable = true)



,letter,labell
0,A,Bike
1,B,Handball
2,C,Bike
3,D,Handball
4,E,Running


In [7]:
from datetime import date

dft = spark.createDataFrame([
    ("A", date(2026, 1, 14), 10.0, 50),
    ("D", date(2026, 1, 15), 20.0, 70),
    ("C", date(2026, 1, 17), 80.0, 30),
    ("B", date(2026, 1, 18), 100.0, 10),
    ("F", date(2026, 1, 18), 80.0, 50),
    ("C", date(2026, 1, 15), 80.0, 60),
    ("F", date(2026, 1, 17), 75.0, 70)
], ["product", "date", "price", "quantity"]
)

dft.printSchema()
dft.toPandas()

root
 |-- product: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: long (nullable = true)



,product,date,price,quantity
0,A,2026-01-14,10.0,50
1,D,2026-01-15,20.0,70
2,C,2026-01-17,80.0,30
3,B,2026-01-18,100.0,10
4,F,2026-01-18,80.0,50
5,C,2026-01-15,80.0,60
6,F,2026-01-17,75.0,70


## Tests with data_provenance_enabled

### Select

In [8]:
with data_provenance_enabled(spark):
    df3 = df.select("*")
df3.toPandas()

,product,date,price,quantity,_provenance_tagg
0,A,2026-01-15,10.0,90,LogicalRDD_8589934592
1,A,2026-01-16,10.0,120,LogicalRDD_17179869184
2,A,2026-01-17,5.0,300,LogicalRDD_34359738368
3,B,2026-01-15,100.0,20,LogicalRDD_42949672960
4,B,2026-01-16,100.0,30,LogicalRDD_60129542144
5,C,2026-01-17,80.0,60,LogicalRDD_68719476736
6,F,2026-01-16,50.0,70,LogicalRDD_77309411328


In [9]:
with data_provenance_enabled(spark):
    df4 = df2.select("letter")
df4.toPandas()

,letter,_provenance_tagg
0,A,LogicalRDD_8589934592
1,B,LogicalRDD_25769803776
2,C,LogicalRDD_42949672960
3,D,LogicalRDD_60129542144
4,E,LogicalRDD_77309411328


In [10]:
# Some tags for df2 are the same as those for df
df.createOrReplaceTempView("sales")
with data_provenance_enabled(spark):
    res = spark.sql("select product from sales")
res.toPandas()

,product,_provenance_tagg
0,A,SubqueryAlias_8589934592
1,A,SubqueryAlias_17179869184
2,A,SubqueryAlias_34359738368
3,B,SubqueryAlias_42949672960
4,B,SubqueryAlias_60129542144
5,C,SubqueryAlias_68719476736
6,F,SubqueryAlias_77309411328


In [11]:
df2.createOrReplaceTempView("category")
with data_provenance_enabled(spark):
    res2 = spark.sql("select letter from category")
res2.toPandas()

,letter,_provenance_tagg
0,A,SubqueryAlias_8589934592
1,B,SubqueryAlias_25769803776
2,C,SubqueryAlias_42949672960
3,D,SubqueryAlias_60129542144
4,E,SubqueryAlias_77309411328


### Join

In [12]:
df.createOrReplaceTempView("sales")
df2.createOrReplaceTempView("category")
with data_provenance_enabled(spark):
    res = spark.sql("select * from sales s join category c on s.product=c.letter").toPandas()

res

,product,date,price,quantity,letter,labell,_provenance_tagg
0,A,2026-01-15,10.0,90,A,Bike,(SubqueryAlias_8589934592 ⊗ SubqueryAlias_8589...
1,A,2026-01-16,10.0,120,A,Bike,(SubqueryAlias_17179869184 ⊗ SubqueryAlias_858...
2,A,2026-01-17,5.0,300,A,Bike,(SubqueryAlias_34359738368 ⊗ SubqueryAlias_858...
3,B,2026-01-15,100.0,20,B,Handball,(SubqueryAlias_42949672960 ⊗ SubqueryAlias_257...
4,B,2026-01-16,100.0,30,B,Handball,(SubqueryAlias_60129542144 ⊗ SubqueryAlias_257...
5,C,2026-01-17,80.0,60,C,Bike,(SubqueryAlias_68719476736 ⊗ SubqueryAlias_429...


In [13]:
# Only the provenance tags of df is taken in account
with data_provenance_enabled(spark):
    df5 = df.select("*").join(df2, df.product == df2.letter)

df5.toPandas()

,product,date,price,quantity,_provenance_tagg,letter,labell
0,A,2026-01-15,10.0,90,LogicalRDD_8589934592,A,Bike
1,A,2026-01-16,10.0,120,LogicalRDD_17179869184,A,Bike
2,A,2026-01-17,5.0,300,LogicalRDD_34359738368,A,Bike
3,B,2026-01-15,100.0,20,LogicalRDD_42949672960,B,Handball
4,B,2026-01-16,100.0,30,LogicalRDD_60129542144,B,Handball
5,C,2026-01-17,80.0,60,LogicalRDD_68719476736,C,Bike


In [14]:
with data_provenance_enabled(spark):
    df6 = df.join(df2, df.product == df2.letter).select("product")

df6.toPandas()

,product,_provenance_tagg
0,A,(LogicalRDD_8589934592 ⊗ LogicalRDD_8589934592)
1,A,(LogicalRDD_17179869184 ⊗ LogicalRDD_8589934592)
2,A,(LogicalRDD_34359738368 ⊗ LogicalRDD_8589934592)
3,B,(LogicalRDD_42949672960 ⊗ LogicalRDD_25769803776)
4,B,(LogicalRDD_60129542144 ⊗ LogicalRDD_25769803776)
5,C,(LogicalRDD_68719476736 ⊗ LogicalRDD_42949672960)


In [15]:
# Same here, only the provenance tags of df are taken in account
with data_provenance_enabled(spark):
    df7 = df.select("*").join(df2, df.product==df2.letter, "outer")
df7.toPandas()

,product,date,price,quantity,_provenance_tagg,letter,labell
0,A,2026-01-15,10.0,90.0,LogicalRDD_8589934592,A,Bike
1,A,2026-01-16,10.0,120.0,LogicalRDD_17179869184,A,Bike
2,A,2026-01-17,5.0,300.0,LogicalRDD_34359738368,A,Bike
3,B,2026-01-15,100.0,20.0,LogicalRDD_42949672960,B,Handball
4,B,2026-01-16,100.0,30.0,LogicalRDD_60129542144,B,Handball
5,C,2026-01-17,80.0,60.0,LogicalRDD_68719476736,C,Bike
6,NaN,None,NaN,NaN,NaN,D,Handball
7,NaN,None,NaN,NaN,NaN,E,Running
8,F,2026-01-16,50.0,70.0,LogicalRDD_77309411328,NaN,NaN


### Where

In [16]:
with data_provenance_enabled(spark):
    df8 = df.select("*").filter("price > 10")
df8.toPandas()

,product,date,price,quantity,_provenance_tagg
0,B,2026-01-15,100.0,20,LogicalRDD_42949672960
1,B,2026-01-16,100.0,30,LogicalRDD_60129542144
2,C,2026-01-17,80.0,60,LogicalRDD_68719476736
3,F,2026-01-16,50.0,70,LogicalRDD_77309411328


In [17]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark):
    res = spark.sql("select product, price from sales where price>10").toPandas()

res

,product,price,_provenance_tagg
0,B,100.0,Filter_42949672960
1,B,100.0,Filter_60129542144
2,C,80.0,Filter_68719476736
3,F,50.0,Filter_77309411328


### Aggregation

In [18]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark):
    res = spark.sql("select sum(quantity) from sales group by product").toPandas()
res

,sum(quantity),_provenance_tagg
0,510,{SubqueryAlias_8589934592 ⊕ SubqueryAlias_1717...
1,50,{SubqueryAlias_60129542144 ⊕ SubqueryAlias_429...
2,60,{SubqueryAlias_68719476736}
3,70,{SubqueryAlias_77309411328}


In [19]:
with data_provenance_enabled(spark):
    df10 = df.groupBy("product").agg({"quantity": "sum"})
df10.toPandas()

,product,sum(quantity),_provenance_tagg
0,A,510,{LogicalRDD_8589934592 ⊕ LogicalRDD_1717986918...
1,B,50,{LogicalRDD_42949672960 ⊕ LogicalRDD_60129542144}
2,C,60,{LogicalRDD_68719476736}
3,F,70,{LogicalRDD_77309411328}


In [20]:
with data_provenance_enabled(spark) :
    df11 = df.withColumn("revenue", df["quantity"] * df["price"]) \
            .groupBy("product") \
            .agg({"revenue": "sum"})
df11.toPandas()

,product,sum(revenue),_provenance_tagg
0,A,3600.0,{LogicalRDD_8589934592 ⊕ LogicalRDD_1717986918...
1,B,5000.0,{LogicalRDD_42949672960 ⊕ LogicalRDD_60129542144}
2,C,4800.0,{LogicalRDD_68719476736}
3,F,3500.0,{LogicalRDD_77309411328}


In [21]:
with data_provenance_enabled(spark):
    df12 = df.groupBy("product").agg({"quantity": "sum"})
df12.toPandas()

,product,sum(quantity),_provenance_tagg
0,A,510,{LogicalRDD_8589934592 ⊕ LogicalRDD_1717986918...
1,B,50,{LogicalRDD_42949672960 ⊕ LogicalRDD_60129542144}
2,C,60,{LogicalRDD_68719476736}
3,F,70,{LogicalRDD_77309411328}


In [22]:
# The distinct does not work as expected
# The result should be like the result of the next exemple
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark) :
    res = spark.sql("select distinct product from sales").toPandas()

res

,product,_provenance_tagg
0,A,SubqueryAlias_8589934592
1,A,SubqueryAlias_17179869184
2,A,SubqueryAlias_34359738368
3,B,SubqueryAlias_42949672960
4,B,SubqueryAlias_60129542144
5,C,SubqueryAlias_68719476736
6,F,SubqueryAlias_77309411328


In [23]:
# Correct result
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark) :
    res = spark.sql("select product from sales group by product").toPandas()
res

,product,_provenance_tagg
0,A,{SubqueryAlias_8589934592 ⊕ SubqueryAlias_1717...
1,B,{SubqueryAlias_60129542144 ⊕ SubqueryAlias_429...
2,C,{SubqueryAlias_68719476736}
3,F,{SubqueryAlias_77309411328}


In [24]:
with data_provenance_enabled(spark):
    df13 = df.select("product").union(dft.select("product"))
df13.toPandas()

,product,_provenance_tagg
0,A,LogicalRDD_8589934592
1,A,LogicalRDD_17179869184
2,A,LogicalRDD_34359738368
3,B,LogicalRDD_42949672960
4,B,LogicalRDD_60129542144
5,C,LogicalRDD_68719476736
6,F,LogicalRDD_77309411328
7,A,LogicalRDD_8589934592
8,D,LogicalRDD_17179869184
9,C,LogicalRDD_34359738368


In [25]:
# The result should be the aggregation of provenance tags grouped by product
df.createOrReplaceTempView("sales1")
dft.createOrReplaceTempView("sales2")

with data_provenance_enabled(spark):
    res = spark.sql("select s1.product from sales1 as s1 union select s2.product from sales2 as s2")
res.toPandas()

,product,_provenance_tagg
0,A,SubqueryAlias_8589934592
1,A,SubqueryAlias_17179869184
2,A,SubqueryAlias_34359738368
3,B,SubqueryAlias_42949672960
4,B,SubqueryAlias_60129542144
5,C,SubqueryAlias_68719476736
6,F,SubqueryAlias_77309411328
7,D,SubqueryAlias_17179869184
8,C,SubqueryAlias_34359738368
9,F,SubqueryAlias_60129542144


In [26]:
# Same exemple without provenance
spark.sql("select s1.product from sales1 as s1 union select s2.product from sales2 as s2").toPandas()

,product
0,A
1,B
2,C
3,F
4,D


### Filter / Sort

In [27]:
# With select statement
with data_provenance_enabled(spark):
    df14 = df.select("*").filter("price > 10").sort("price").join(df2, df.product == df2.letter)
df14.toPandas()

,product,date,price,quantity,_provenance_tagg,letter,labell
0,B,2026-01-15,100.0,20,LogicalRDD_42949672960,B,Handball
1,B,2026-01-16,100.0,30,LogicalRDD_60129542144,B,Handball
2,C,2026-01-17,80.0,60,LogicalRDD_68719476736,C,Bike


In [28]:
# Without select statement
with data_provenance_enabled(spark):
    df15 = df.filter("price > 10").sort("price").join(df2, df.product == df2.letter)
df15.toPandas()

,product,date,price,quantity,letter,labell,_provenance_tagg
0,B,2026-01-16,100.0,30,B,Handball,(Sort_3 ⊗ LogicalRDD_25769803776)
1,B,2026-01-15,100.0,20,B,Handball,(Sort_2 ⊗ LogicalRDD_25769803776)
2,C,2026-01-17,80.0,60,C,Bike,(Sort_1 ⊗ LogicalRDD_42949672960)


In [29]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark):
    res = spark.sql("select * from sales order by date desc").toPandas()

res

,product,date,price,quantity,_provenance_tagg
0,A,2026-01-17,5.0,300,SubqueryAlias_34359738368
1,C,2026-01-17,80.0,60,SubqueryAlias_68719476736
2,A,2026-01-16,10.0,120,SubqueryAlias_17179869184
3,B,2026-01-16,100.0,30,SubqueryAlias_60129542144
4,F,2026-01-16,50.0,70,SubqueryAlias_77309411328
5,A,2026-01-15,10.0,90,SubqueryAlias_8589934592
6,B,2026-01-15,100.0,20,SubqueryAlias_42949672960


In [30]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark):
    res = spark.sql("select * from sales where price > 10").toPandas()

res

,product,date,price,quantity,_provenance_tagg
0,B,2026-01-15,100.0,20,Filter_42949672960
1,B,2026-01-16,100.0,30,Filter_60129542144
2,C,2026-01-17,80.0,60,Filter_68719476736
3,F,2026-01-16,50.0,70,Filter_77309411328
